In [237]:
# --- Step 1: Load Data from SQLite and Create Documents ---
import sqlite3
from langchain.schema import Document

sqlite_file = "./courses.sqlite"
table_name = "merged_courses"
content_columns = [
    "title", "instructor", "learning_obj", "course_contents", "prerequisites",
    "readings", "applicability", "workload", "credits", "evaluation", 
    "time", "duration", "course_type"
]

def load_db(sqlite_file, table_name, content_columns):
    conn = sqlite3.connect(sqlite_file)
    cursor = conn.cursor()
    
    # Query all rows
    query = f"SELECT * FROM {table_name}"
    cursor.execute(query)
    
    rows = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]
    conn.close()

    # Combine content columns into `page_content` and include metadata
    documents = [
        Document(
            page_content=" ".join(str(row[column_names.index(col)]) for col in content_columns),
            metadata={col: row[column_names.index(col)] for col in column_names}
        )
        for row in rows
    ]
    return documents

documents = load_db(sqlite_file, table_name, content_columns)
print(documents[:2])

[Document(metadata={'rowid': 1, 'file_loc': 'E:\\HSM\\project_hmi\\webappcoursespot\\courses\\zqm_modul\\en\\Automotive Technology Mgmt_MD.pdf', 'title': 'Automotive Technology Management', 'instructor': 'Prof Dr Michael Dornieden', 'learning_obj': 'Identify relevant market framework conditions of carmakers\r▪ Realize the economic importance of the automotive industry\r▪ Know automotive key figures in respect to particular countries\r▪ Categorize the product portfolio of automotive suppliers\r▪ Examine and understand future trends of automotive markets and\rresource requirements of carmakers (e.g. electromobility)\r▪ Critically evaluate strategic alliances between carmakers currently in\rpractical company use and develop an understanding of how they can\rbe improved in order to reach the desired organizational goals\r▪ Know theoretical foundations of product lifecycle management\r▪ Characterize the six phases of the Generic Product Development\rProcess and demonstrate its application t

In [238]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=150, add_start_index=True
)
all_splits = text_splitter.split_documents(documents)

In [239]:
# Step 3: Persist the vector store
import os
from langchain_community.embeddings import OllamaEmbeddings
# from langchain_chroma import Chroma
from langchain_community.vectorstores.chroma import Chroma
import logging

# Set the path for the vector store
VECTORSTORE_DIR = "./chroma_vectorstore"
def initialize_vectorstore():
    if not os.path.exists(VECTORSTORE_DIR):
        # If the directory doesn't exist, create and persist the vector store
        local_embeddings = OllamaEmbeddings(model="all-minilm")
        vectorstore = Chroma.from_documents(
            documents=all_splits,
            embedding=local_embeddings,
            persist_directory=VECTORSTORE_DIR
        )
        vectorstore.persist()
        logging.info("Vector store created and persisted to disk.")
    else:
        logging.info("Vector store already exists. Skipping embedding computation.")

# Initialize the vector store (only computes embeddings if not already persisted)
initialize_vectorstore()

In [240]:
# Load the vector store for querying
def load_vectorstore():
    local_embeddings = OllamaEmbeddings(model="all-minilm")
    return Chroma(
        persist_directory=VECTORSTORE_DIR,
        embedding_function=local_embeddings
    )

vectorstore = load_vectorstore()

# print(vectorstore)

In [241]:

from langchain_ollama.llms import OllamaLLM

In [242]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [243]:
from rapidfuzz import process
import logging

# Step 4: Define a function for query correction using fuzzy matching
# def correct_query(query, documents, top_n=1):
#     # Extract titles or key text from documents for fuzzy matching
#     document_titles = [doc.metadata.get("title", "") for doc in documents]
    
#     # Use fuzzy matching to find the closest matches
#     best_matches = process.extract(query, document_titles, limit=top_n)
#     corrected_query = best_matches[0][0] if best_matches else query  # Take the best match
#     return corrected_query

# def correct_query(query, documents, content_columns, top_n=1):
#     """
#     Fuzzy-match the user's query against a combined metadata string from each document.

#     :param query: User's raw query string
#     :param documents: List of Document objects
#     :param content_columns: List of metadata keys to include in the fuzzy match
#     :param top_n: How many best matches to return
#     :return: The combined metadata string of the best match (or original query if no matches found)
#     """
#     # Build a list of combined metadata strings (one per document) 
#     combined_strings = []
#     for doc in documents:
#         # Combine relevant metadata fields into a single string
#         # Convert each field to string to avoid errors in case of numeric fields
#         combined_text = " ".join(
#             str(doc.metadata.get(col, "")) for col in content_columns
#         )
#         combined_strings.append(combined_text)

#     # Use fuzzy matching to find the closest matches to the user's query
#     best_matches = process.extract(query, combined_strings, limit=top_n)
    
#     # best_matches is a list of tuples: (matched_string, score, index)
#     if best_matches:
#         best_match_index = best_matches[0][2]  # index of the best match in `combined_strings`
#         best_match_text = combined_strings[best_match_index]
#         return best_match_text
#     else:
#         # If no match is found, just return the original query
#         return query
def fuzzy_match_docs(query, documents, content_columns, top_n=5):
    """
    Fuzzy-match the user's query against a combined metadata string from each document.
    Returns the top_n matching Document objects (plus their score, if needed).
    """
    combined_strings = []
    for doc in documents:
        # Combine relevant metadata fields (convert everything to string just in case)
        # combined_text = " ".join(str(doc.metadata.get(col, "")) for col in content_columns)
        combined_text = " ".join([
            (str(doc.metadata.get("title", "")) + " ") * 5,  # repeat thrice
            str(doc.metadata.get("instructor", "")),
            str(doc.metadata.get("learning_obj", "")),
            str(doc.metadata.get("course_contents", "")),
            str(doc.metadata.get("prerequisites", "")),
            str(doc.metadata.get("applicability", "")),
            str(doc.metadata.get("course_type", ""))
        ])
        combined_strings.append(combined_text)

    # Get the top_n best matches
    best_matches = process.extract(query, combined_strings, limit=top_n)

    # Build results: (Document, score)
    results = []
    for match_str, score, idx in best_matches:
        matched_doc = documents[idx]
        results.append({"doc": matched_doc, "score": score, "matched_text": match_str})

    return results

In [244]:
def hybrid_retrieve(question: str, 
                    documents: list, 
                    content_columns: list, 
                    retriever,
                    top_n: int = 5):
    # 1) Fuzzy matches
    fuzzy_results = fuzzy_match_docs(query=question,
                                     documents=documents,
                                     content_columns=content_columns,
                                     top_n=top_n)  # returns list of dicts

    # 2) Vector results
    vector_results = retriever.get_relevant_documents(question)  # returns list of Documents

    # Turn fuzzy docs into a list of Documents
    fuzzy_docs = [res["doc"] for res in fuzzy_results]

    # 3) Combine them (avoid duplication by using a set of page_content or IDs)
    final_docs = []
    seen_contents = set()

    # Add fuzzy docs
    for fd in fuzzy_docs:
        if fd.page_content not in seen_contents:
            seen_contents.add(fd.page_content)
            final_docs.append(fd)

    # Add vector docs
    for vd in vector_results:
        if vd.page_content not in seen_contents:
            seen_contents.add(vd.page_content)
            final_docs.append(vd)

    # Return a combined list of Documents
    return final_docs


In [245]:
def find_it_security_courses(query, documents, content_columns):
    # Get top-5 fuzzy matches
    top_matches = fuzzy_match_docs(query, documents, content_columns, top_n=5)
    
    # Print them for debugging
    print("Top 5 fuzzy matches:")
    for m in top_matches:
        doc = m["doc"]
        score = m["score"]
        print(f"  - Title: {doc.metadata.get('title','UNKNOWN')} | Score: {score}")

    # Check if the doc titled "IT Security" is among them
    it_security_docs = [
        m["doc"] for m in top_matches 
        if m["doc"].metadata.get("title", "").lower().startswith("it security")
    ]
    
    if it_security_docs:
        print("\nFound an IT Security document in the top matches!")
        for doc in it_security_docs:
            print("   =>", doc.metadata.get("title"))
    else:
        print("\nNo exact 'IT Security' doc found in the top matches.")


In [246]:
llm = OllamaLLM(model="llama3.1")

In [247]:
content_columns_final = [
    "title", "instructor", "learning_obj", "course_contents", "prerequisites",
    "readings", "applicability", "course_type"
]
# Define the RAG application class
class RAGApplications:
    def __init__(self, documents, llm, content_columns):
        """
        :param documents: A list of langchain.schema.Document objects
        :param llm: An LLM instance (e.g. OllamaLLM)
        :param content_columns: List of metadata fields (strings) to fuzzy-match against
        """
        self.documents = documents
        self.llm = llm
        self.content_columns = content_columns
    
    def run(self, question):
        # Log the incoming question
        logging.info(f"Original Query: {question}")

        vector_results = self.retriever.get_relevant_documents(question)

        itsec_result = find_it_security_courses(
            query=question, 
            documents=self.documents, 
            content_columns=self.content_columns
        )

        print(itsec_result, "it sec result")

        # --- Step A: Fuzzy match top 5 docs ---
        matched_docs = fuzzy_match_docs(
            query=question,
            documents=self.documents,
            content_columns=self.content_columns,
            top_n=5
        )

        print(matched_docs, "matched docs")

        # --- Step B: Build a combined context from these matched docs ---
        # context = "\n\n".join(doc.page_content for (doc, _) in matched_docs)
        context = "\n\n".join(
            match["doc"].page_content for match in matched_docs
        )

        # --- Step C: Create the prompt for the LLM ---
        prompt = f"""
You are an AI assistant with knowledge of the following context:
{context}

Question: {question}

Instructions:
1. If the question asks for a list of courses, provide them in bullet points with short descriptions.
2. If the question asks for numeric data (like counts, credits, etc.), be precise and mention how you got it from the context.
3. If something isn't mentioned in the context, say "I'm not sure based on the provided context."

Answer:
"""
        # --- Step D: Invoke the LLM ---
        response = self.llm.invoke(prompt)
        
        return {
            "response": response,
            "context": context
        }


In [248]:
#############################
# 3. RAG APPLICATION (VECTOR ONLY)
#############################
class RAGApplication:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    def run(self, question: str):
        logging.info(f"User query: {question}")

        # Retrieve top matches from the vector store
        relevant_docs = self.retriever.get_relevant_documents(question)

        # Build a combined context
        context = "\n\n".join(doc.page_content for doc in relevant_docs)

        # Create a prompt for the LLM
        prompt = f"""
You are an AI assistant with knowledge of the following context:
{context}

The user asked: "{question}"

Instructions:
1. Use only the provided context to answer the question.
2. If the context does not explicitly contain the information, say "I'm not sure based on the context."
3. If asked for a list of courses, list them in bullet points with short descriptions.

Answer:
"""

        # Invoke the LLM
        response = self.llm.invoke(prompt)

        return {"response": response, "context": context}


In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS, cross_origin


# # Initialize Flask app
# app = Flask(__name__)
# CORS(app, support_credentials=True)

# # Initialize logger
# logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# @app.route("/ask", methods=["POST"])
# @cross_origin(supports_credentials=True)
# def ask():
#     """
#     Endpoint to handle user questions and return responses from the RAG application.
#     """
#     try:
#         # Parse JSON data from request
#         data = request.json
#         if not data or "query" not in data:
#             logging.warning("No query provided in the request.")
#             return jsonify({"error": "No query provided"}), 400

#         question = data["query"].strip()
#         if not question:
#             logging.warning("Empty query provided in the request.")
#             return jsonify({"error": "Query cannot be empty"}), 400

#         # Process the question using RAGApplication
#         rag_application = RAGApplication(documents, llm, content_columns_final)
#         answer = rag_application.run(question)

#         logging.info(f"Question: {question} | Answer: {answer}")
#         return jsonify({
#             "response": answer["response"],
#             "context": answer["context"]
#         })

#     except Exception as e:
#         logging.error(f"Error occurred: {str(e)}", exc_info=True)
#         return jsonify({'error': "An internal error occurred. Please try again later."}), 500

# # Run the app
# if __name__ == "__main__":
#     app.run(debug=True, host='0.0.0.0', port=5050, use_reloader=False)



# New from chat gpt
#############################
# 4. FLASK APP
#############################
app = Flask(__name__)
CORS(app, support_credentials=True)

llm = OllamaLLM(model="llama3.1")

# We'll instantiate RAGApplication with our vector retriever
rag_app = RAGApplication(retriever, llm)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

@app.route("/ask", methods=["POST"])
@cross_origin(supports_credentials=True)
def ask():
    try:
        data = request.json
        if not data or "query" not in data:
            return jsonify({"error": "No query provided"}), 400

        question = data["query"].strip()
        if not question:
            return jsonify({"error": "Query cannot be empty"}), 400

        # Use the RAG application
        result = rag_app.run(question)
        logging.info(f"Question: {question} | Answer: {result}")

        return jsonify({
            "response": result["response"],
            "context": result["context"]
        })

    except Exception as e:
        logging.error(f"Error occurred: {str(e)}", exc_info=True)
        return jsonify({"error": "Internal error"}), 500

if __name__ == "__main__":
    app.run(debug=True, host='0.0.0.0', port=5050, use_reloader=False)



 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5050
 * Running on http://192.168.2.33:5050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 01:09:19] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 01:11:39] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 01:14:13] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 01:15:15] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 08:07:46] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 08:09:25] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 08:29:04] "OPTIONS /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 08:30:57] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jan/2025 08:35:30] "OPTIONS /ask HTTP/1.1" 200 -
IN

In [ ]:

        # Get the answer from the language model
        # response = llm.invoke(f"""When you will get any query then please first try to find in our vector databse. This is very important because different kind of question can  be asked. Answer the question according to the context given very briefly:
        #         Question: {question}.
        #         Context: {context}
        #         """)
        # prompt = f"""
        #     You are an AI assistant with access to a knowledge base containing course information. 
        #     You have access to context (excerpts from the database). 
        #     Your job is to answer the user’s question as accurately as possible using ONLY the provided context. 
        #     If the answer is not explicitly stated in the context, say you are unsure.

        #     Context:
        #     {context}

        #     User's question: {question}

        #     Instructions:
        #     1. Provide a clear, concise answer.
        #     2. Where relevant, include short bullet points or lists (for example, to list available courses).
        #     3. If you are missing information or if the context doesn't mention it, say "I'm not sure based on the provided context."

        #     Now please provide your best possible answer:
        #     """